# 01 · Exploración inicial de los datos brutos
Notebook de pruebas: primer contacto con `companies` y `rounds` antes de decidir la limpieza.

In [1]:
import sys, warnings
sys.path.append("..")          # para poder importar src/utils desde src/notebooks
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
DATA_DIR = "../data"

In [2]:
companies = pd.read_csv(f'{DATA_DIR}/raw/companies.csv.gz')
rounds = pd.read_csv(f'{DATA_DIR}/raw/rounds.csv.gz')
companies.shape, rounds.shape

((66368, 14), (114949, 12))

In [3]:
companies.sample(5, random_state=0)

,permalink,name,homepage_url,category_list,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,first_funding_at,last_funding_at
20403,/organization/findery,Findery,http://findery.com,Mobile,9500000,operating,USA,CA,SF Bay Area,San Francisco,2,2011-01-01,2011-02-16,2012-02-16
23284,/organization/gloss48,Gloss48,http://gloss48.com,E-Commerce|Marketplaces,800000,operating,USA,MA,Boston,Boston,1,2011-09-15,2013-06-01,2013-06-01
55435,/organization/superdata-research,SuperData Research,http://www.superdataresearch.com,Games|Market Research|MMO Games|Mobile|Video G...,7377578,operating,USA,NY,New York City,New York,4,2009-08-01,1999-04-01,2010-07-01
37019,/organization/mophie,Mophie,http://www.mophie.com,Hardware + Software,1000000,acquired,USA,CA,"Orange County, California",Tustin,2,2005-05-24,2006-08-01,2010-08-02
45755,/organization/prynt,Prynt,http://www.pryntcases.com,Consumer Electronics|Mobile|Photography,2200000,operating,USA,CA,SF Bay Area,San Francisco,3,2014-06-14,2014-07-14,2015-04-10


In [4]:
companies.isna().mean().sort_values(ascending=False).mul(100).round(1)

founded_at           22.9
state_code           12.9
region               12.1
city                 12.1
country_code         10.5
homepage_url          7.6
category_list         4.7
first_funding_at      0.0
name                  0.0
permalink             0.0
status                0.0
funding_total_usd     0.0
funding_rounds        0.0
last_funding_at       0.0
dtype: float64

`funding_total_usd` viene como texto. ¿Qué valores no numéricos tiene?

In [5]:
no_num = pd.to_numeric(companies['funding_total_usd'], errors='coerce').isna()
companies.loc[no_num, 'funding_total_usd'].value_counts()

funding_total_usd
-    12785
Name: count, dtype: int64

In [6]:
companies['status'].value_counts()

status
operating    53034
closed        6238
acquired      5549
ipo           1547
Name: count, dtype: int64

In [7]:
rounds['funding_round_type'].value_counts()

funding_round_type
venture                  55494
seed                     30524
debt_financing            6895
angel                     6094
undisclosed               4897
equity_crowdfunding       3257
private_equity            2285
grant                     2200
convertible_note          1817
post_ipo_equity            638
product_crowdfunding       410
non_equity_assistance      191
post_ipo_debt              152
secondary_market            95
Name: count, dtype: int64

In [8]:
rounds['funding_round_code'].value_counts(dropna=False)

funding_round_code
NaN    83809
A      14747
B       8713
C       4476
D       2025
E        811
F        283
G         73
H         12
Name: count, dtype: int64

Fechas raras: años de fundación imposibles.

In [9]:
anio = pd.to_datetime(companies['founded_at'], errors='coerce').dt.year
anio.describe(), companies.loc[(anio < 1900) | (anio > 2015), ['name', 'founded_at']]

(count    51147.000000
 mean      2007.745205
 std         12.146384
 min       1015.000000
 25%       2006.000000
 50%       2010.000000
 75%       2013.000000
 max       2914.000000
 Name: founded_at, dtype: float64,
                                      name  founded_at
 1879                                 AG&P  1899-12-31
 2353              Alcorn State University  1871-01-01
 2952   American Museum of Natural History  1869-01-01
 2959                   American Red Cross  1881-05-01
 3736                      Appleton Coated  1889-01-01
 ...                                   ...         ...
 63949              Wilberforce University  1856-01-01
 64253                                WOLF  1843-01-01
 64340     Worcester Polytechnic Institute  1865-01-01
 64679                   Xavier University  1831-01-01
 66051                Zions Bancorporation  1873-01-01
 
 [114 rows x 2 columns])

¿Coincide `funding_rounds` de companies con el nº de filas en rounds?

In [10]:
n_rondas = rounds.groupby('company_permalink').size()
comp = companies.set_index('permalink')['funding_rounds']
(comp == n_rondas.reindex(comp.index)).mean()

np.float64(0.999472637415622)

Primera prueba rápida de la hipótesis (sin limpiar, toda la muestra):

In [11]:
f = pd.to_numeric(companies['funding_total_usd'], errors='coerce')
pd.crosstab(pd.qcut(f, 5), companies['status'], normalize='index').round(3)

status,acquired,closed,ipo,operating
funding_total_usd,,,,
"(0.999, 200000.0]",0.016,0.116,0.001,0.867
"(200000.0, 1000000.0]",0.041,0.101,0.004,0.854
"(1000000.0, 3844984.0]",0.075,0.081,0.011,0.833
"(3844984.0, 14633143.8]",0.135,0.090,0.029,0.747
"(14633143.8, 30079503000.0]",0.180,0.071,0.089,0.660


**Notas para la limpieza:** `'-'` → NaN; fechas fuera de rango → NaT; quitar columnas duplicadas de `rounds`; ojo con las startups jóvenes (no han tenido tiempo de cerrar/venderse).